# HMM（隱藏馬可夫模型，選讀）（1960s）

- **核心原理**：隱藏狀態＋轉移機率
- **變體**：Gaussian HMM（連續觀測）、Input-Output HMM、階層 HMM；監督式 HMM 標註器（詞性、NER）
- **推薦資料集**：選讀（README 沒指定）→ 本章用兩份：
  1. **「不老實的賭場」**（NumPy 模擬）：已知真正的參數與隱藏狀態，用來驗證每個演算法；
  2. **CoNLL-2003 NER**（下一章 CRF 的資料）：用「數數」訓練一個監督式 HMM，當第 21 章的 baseline。

前面所有章節都假設樣本**獨立**：洗牌不影響結果。HMM 是第一個處理**序列**的模型：

- 每個時間點有一個**看不到**的狀態 $z_t$（賭場用的是公正骰子還是灌鉛骰子；這個字是不是人名的一部分），
- 狀態按照**轉移機率** $A$ 一步一步變（馬可夫：只看上一步），
- 每個狀態按照**發射機率** $B$ 產生**看得到**的觀測 $x_t$（擲出的點數；這個字本身）。

三個經典問題（Rabiner 1989）：

| 問題 | 問什麼 | 演算法 |
| --- | --- | --- |
| 評估 | 這串觀測的機率 $P(x_{1:T})$ 是多少？ | forward |
| 解碼 | 最可能的隱藏狀態序列是什麼？ | Viterbi（整條路徑）／forward-backward（每一步的後驗） |
| 學習 | 只有觀測，怎麼估 $\pi, A, B$？ | Baum-Welch（= EM，第十五章） |

這一章的重點：**為什麼不能暴力列舉（$K^T$ 條路徑）、動態規劃把它變成 $O(TK^2)$、為什麼要在 log 空間算、Viterbi 和後驗解碼的差別、Baum-Welch 就是 EM、以及監督式 HMM 標註器的極限（生成式模型、只能看「這個字」）—— 這是下一章 CRF 要解決的。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3D、5 節開頭另有各自的符號表或名詞表。

**資料**：

- 賭場：每次擲一顆骰子，觀測 $x_t \in \{0, \dots, 5\}$（代表點數 1–6）。隱藏狀態 $z_t \in \{0, 1\}$ = {F（公正）, L（灌鉛：6 點的機率 0.5）}。
- CoNLL-2003：每句話是一串英文單字，標籤是 9 種 BIO 標籤（`O`、`B-PER`、`I-PER`、`B-ORG`…）。第 5 節再介紹。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 賭場的值 |
| --- | --- | --- |
| $T$、$K$、$V$ | 序列長度、狀態數、觀測種類數 | 2,000、2、6 |
| $\pi_k$ | 初始機率 $P(z_1 = k)$ | $(0.5, 0.5)$ |
| $A_{jk}$ | 轉移機率 $P(z_t = k \mid z_{t-1} = j)$（每一**列**加總 1） | $\begin{pmatrix}0.95 & 0.05\\ 0.10 & 0.90\end{pmatrix}$ |
| $B_{kv}$ | 發射機率 $P(x_t = v \mid z_t = k)$（每一列加總 1） | F：$1/6$ 各；L：$0.1 \times 5$、6 點 $0.5$ |
| $\alpha_t(k)$ | forward：$P(x_{1:t}, z_t = k)$ | — |
| $\beta_t(k)$ | backward：$P(x_{t+1:T} \mid z_t = k)$ | — |
| $\gamma_t(k)$ | 後驗 $P(z_t = k \mid x_{1:T})$ | — |
| $\delta_t(k)$ | Viterbi：走到 $(t, k)$ 的最佳路徑機率 | — |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第六章 | 樸素貝氏 | 生成式模型：學 $P(x \mid y)$ 和 $P(y)$，用貝氏定理反推 $P(y \mid x)$；拉普拉斯平滑（$+\alpha$）；連乘會 underflow → 取 log，`logsumexp`。 |
| 第十五章 | GMM（`01-classic-ml/15-gmm`） | EM：E-step 算責任 $\gamma_{ik}$（後驗）、M-step 用 $\gamma$ 加權重新估參數；log-likelihood 每一步都不會下降；有局部最佳、標籤互換（label switching）；用 BIC 選元件數。 |
| 第 14、15 章 | PCA、t-SNE／UMAP | 把每個樣本當成獨立的點；不看順序。 |
| 第 21 章 | CRF | 判別式的序列模型：直接學 $P(z \mid x)$，可以用任意重疊的特徵；用同一份 CoNLL-2003、同一個成績檔比較。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 連鎖律**：$P(a, b, c) = P(a)\,P(b \mid a)\,P(c \mid a, b)$。

**2. 馬可夫性質**：$P(z_t \mid z_{1:t-1}) = P(z_t \mid z_{t-1})$；**觀測獨立**：$P(x_t \mid z_{1:T}, x_{\ne t}) = P(x_t \mid z_t)$。所以聯合機率是
$$P(x_{1:T}, z_{1:T}) = \pi_{z_1} B_{z_1 x_1}\prod_{t=2}^T A_{z_{t-1} z_t} B_{z_t x_t}$$
例：$z = (L, L)$、$x = (6, 6)$ → $0.5 \times 0.5 \times 0.9 \times 0.5 = 0.1125$。

**3. 邊際化**：$P(x) = \sum_z P(x, z)$ —— 要加總 $K^T$ 條路徑（$2^{2000}$！）。

**4. 動態規劃**：把「所有路徑的總和」拆成「走到第 $t$ 步、停在狀態 $k$ 的所有路徑的總和」，下一步只需要上一步的 $K$ 個數。

**5. log-sum-exp**（第六章）：$\ln\sum_j e^{a_j} = m + \ln\sum_j e^{a_j - m}$，$m = \max_j a_j$。乘法變加法、加法變 `logsumexp`。

**6. EM**（第十五章）：E-step 算隱藏變數的後驗（這裡是 $\gamma_t(k)$ 和 $\xi_t(j, k)$），M-step 用它們「加權數數」。

#### 程式

**7. `logsumexp(a, axis=...)`**：`scipy.special.logsumexp` 有很多輸入檢查，對 $2 \times 2$ 的小陣列一次要 100 多 µs；forward 每一步呼叫一次、Baum-Welch 跑幾百輪，所以第一格自己寫一個簡化版（同樣的公式），快約 15 倍。
**8. 廣播**：`la[t - 1][:, None] + logA` 是 $(K, K)$，第 $(j, k)$ 個 = $\ln\alpha_{t-1}(j) + \ln A_{jk}$；對 `axis=0`（$j$）做 logsumexp → 長度 $K$（每個 $k$ 一個）。
**9. `argmax` 與回溯指標**：Viterbi 每一步記住「從哪一個 $j$ 來最好」（`bp[t, k]`），最後從終點倒著走回去。
**10. `np.add.at(B, (z, x), 1)`**：在 `B[z[i], x[i]]` 上各加 1（同一格出現多次也會累加；`B[z, x] += 1` 不會）。

---
## 1. 資料與探索性資料分析（EDA）：不老實的賭場

賭場大部分時間用公正骰子，偶爾（每次 5%）偷偷換成灌鉛骰子，灌鉛時每次 10% 換回來。模擬 2,000 次，**我們知道答案**（每一次用的是哪顆骰子），所以可以檢查每個演算法。

In [ ]:
import itertools
import time

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


def logsumexp(a: np.ndarray, axis: int | None = None) -> np.ndarray | float:
    """ln Σ exp(a)（前置複習 5、7）。功能同 scipy.special.logsumexp，但對小陣列快約 15 倍（這章要呼叫幾十萬次）。"""
    m = np.max(a, axis=axis, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.0)  # 整欄都是 −inf（機率 0）時避免 −inf − (−inf) = nan
    with np.errstate(divide="ignore"):
        out = m + np.log(np.exp(a - m).sum(axis=axis, keepdims=True))
    return out.item() if axis is None else np.squeeze(out, axis=axis)


STATES = ["F", "L"]
pi_true = np.array([0.5, 0.5])
A_true = np.array([[0.95, 0.05], [0.10, 0.90]])
B_true = np.array([[1 / 6] * 6, [0.1] * 5 + [0.5]])


def sample_hmm(T: int, pi: np.ndarray, A: np.ndarray, B: np.ndarray, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    """回傳 (隱藏狀態 z (T,), 觀測 x (T,))。"""
    z = np.zeros(T, dtype=int)
    x = np.zeros(T, dtype=int)
    z[0] = rng.choice(len(pi), p=pi)
    for t in range(T):
        if t > 0:
            z[t] = rng.choice(len(pi), p=A[z[t - 1]])
        x[t] = rng.choice(B.shape[1], p=B[z[t]])
    return z, x


z_true, x_obs = sample_hmm(2000, pi_true, A_true, B_true, np.random.default_rng(0))
T = len(x_obs)
print("灌鉛的比例：", z_true.mean().round(3), "（理論穩態：0.05 / (0.05 + 0.10) = 0.333）")
print("點數 1–6 出現次數：", np.bincount(x_obs, minlength=6).tolist())
print("在 F 時：", np.bincount(x_obs[z_true == 0], minlength=6).tolist(), "；在 L 時：", np.bincount(x_obs[z_true == 1], minlength=6).tolist())

runs = np.diff(np.flatnonzero(np.diff(np.r_[-1, z_true, -1]) != 0))  # 每一段連續相同狀態的長度
run_state = z_true[np.r_[0, np.cumsum(runs)[:-1]]]
print("F 連續段的平均長度：", runs[run_state == 0].mean().round(1), "（理論 1/0.05 = 20）；L：", runs[run_state == 1].mean().round(1), "（理論 1/0.10 = 10）")

fig, axes = plt.subplots(2, 1, figsize=(15, 4), sharex=True)
axes[0].plot(z_true[:300], drawstyle="steps-mid")
axes[0].set_yticks([0, 1], STATES)
axes[0].set_title("hidden state (first 300 rolls)")
axes[1].scatter(np.arange(300), x_obs[:300] + 1, c=z_true[:300], cmap="coolwarm", s=8)
axes[1].set_ylabel("roll")
plt.show()

In [ ]:
# 順序重要嗎？「是不是 6」的自我相關（lag 1–30）：原始序列 vs 洗牌後
six = (x_obs == 5).astype(float)


def autocorr(s: np.ndarray, max_lag: int = 30) -> np.ndarray:
    s = s - s.mean()
    return np.array([(s[:-k] * s[k:]).mean() / s.var() for k in range(1, max_lag + 1)])


fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(np.arange(1, 31) - 0.2, autocorr(six), width=0.4, label="original")
ax.bar(np.arange(1, 31) + 0.2, autocorr(np.random.default_rng(0).permutation(six)), width=0.4, label="shuffled")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("lag")
ax.set_ylabel("autocorrelation of 'is six'")
ax.legend()
plt.show()

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 整體來看，6 點出現的比例是多少？如果只看「點數的直方圖」（例如用第十五章的混合模型），
#     能估出「有兩顆骰子、灌鉛的那顆 6 點機率 0.5」嗎？能知道「什麼時候」換骰子嗎？
# 答：
#
# (2) 自我相關：原始序列在 lag 1、5、20 的自相關是多少？洗牌之後呢？
#     這個差別就是「順序」帶來的資訊 —— 哪一個參數（π、A、B）在描述它？
# 答：
#
# (3) F、L 連續段的平均長度跟理論值（1 / 離開的機率）接近嗎？為什麼是 1 / p？（提示：幾何分布）
# 答：

---
## 2. Baseline：每一步獨立判斷（不看前後文）

假裝參數都知道（$\pi$ 用穩態 1/3、$B$），但**不用轉移機率**：每一次擲骰子各自用貝氏定理（第六章）判斷是 F 還是 L。再試一個「看前後文」的土法：**滑動視窗**裡 6 點的比例超過門檻就說是 L。

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| 單步貝氏 | $P(L \mid x_t) \propto P(L)\,B_{L, x_t}$：只有擲出 6 才會判成 L |
| 滑動視窗 | 前後各 $w$ 次（共 $2w + 1$ 次）裡 6 點的比例 |

In [ ]:
prior_L = 1 / 3
post_single = prior_L * B_true[1, x_obs] / (prior_L * B_true[1, x_obs] + (1 - prior_L) * B_true[0, x_obs])
pred_single = (post_single > 0.5).astype(int)

rows = [{"method": "always F", "accuracy": (z_true == 0).mean()},
        {"method": "single-roll Bayes", "accuracy": (pred_single == z_true).mean()}]
for w in [5, 10, 20]:
    frac6 = np.convolve(six, np.ones(2 * w + 1) / (2 * w + 1), mode="same")
    for thr in [0.25, 0.33]:
        rows.append({"method": f"window ±{w}, six-fraction > {thr}", "accuracy": ((frac6 > thr).astype(int) == z_true).mean()})
base_tbl = pd.DataFrame(rows).set_index("method")
display(base_tbl.round(4))

In [ ]:
# TODO 2（問答）
#
# (1) 單步貝氏：P(L | 擲出 6) 和 P(L | 擲出 1) 各是多少？它的準確率比「永遠猜 F」好嗎？為什麼幾乎沒用？
# 答：
#
# (2) 滑動視窗：哪個 w、哪個門檻最好？視窗太小、太大各有什麼問題？
#     這個方法有兩個「手調」的超參數 —— HMM 會用哪些參數取代它們？
# 答：

---
## 3. 手寫模型

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | forward：$P(x_{1:T})$（log 空間） | 3 |
| 3B | backward 與後驗 $\gamma$ | 4、5 |
| 3C | Viterbi：最可能的路徑 | 6 |
| 3D | Baum-Welch（EM）：只有觀測，學參數 | 8 |
| 3E | 實驗：解碼方式、資料量、局部最佳、狀態數 | — |

### 3A. forward 演算法

$\alpha_t(k) = P(x_{1:t}, z_t = k)$：「看到前 $t$ 個觀測，而且第 $t$ 步在狀態 $k$」的機率（把所有走到那裡的路徑加起來）。

$$\alpha_1(k) = \pi_k B_{k, x_1}, \qquad \alpha_t(k) = \Big[\sum_j \alpha_{t-1}(j) A_{jk}\Big] B_{k, x_t}, \qquad P(x_{1:T}) = \sum_k \alpha_T(k)$$

每一步 $O(K^2)$，總共 $O(TK^2)$ —— 而不是 $K^T$。

**log 空間**：$\alpha$ 是很多個小於 1 的數連乘，$T = 2000$ 時大約 $10^{-1500}$，float64 最小只到 $10^{-308}$ → 變成 0。改存 $\ln\alpha$：
$$\ln\alpha_t(k) = \text{logsumexp}_j\big(\ln\alpha_{t-1}(j) + \ln A_{jk}\big) + \ln B_{k, x_t}$$

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $\ln\pi$、$\ln A$、$\ln B$ | 參數取 log | $(K,)$、$(K, K)$、$(K, V)$ | `lpi`、`lA`、`lB` |
| $\ln\alpha$ | forward（log） | $(T, K)$ | `la` |
| $\ln B_{\cdot, x_t}$ | 第 $t$ 步各狀態的發射 log 機率 | $(K,)$ | `lB[:, x[t]]` |

**下標公式**：
- TODO 3：$\ln\alpha_t(k) = \text{logsumexp}_j(\ln\alpha_{t-1}(j) + \ln A_{jk}) + \ln B_{k, x_t}$ → `logsumexp(la[t - 1][:, None] + lA, axis=0) + lB[:, x[t]]`（前置複習 8）。

#### 最小例子：TODO 3（3 次擲骰子）

$x = (6, 6, 1)$，用真正的參數。

1. **正確答案**：暴力列舉 $2^3 = 8$ 條路徑，每條用前置複習 2 的公式，加起來。最大的一條是 LLL（0.010125），總和 $P(x) \approx 0.015058$，$\ln P(x) \approx -4.1959$。
2. **積木**：一般空間的 forward：$\alpha_1 = (0.0833, 0.25)$、$\alpha_2 = (0.017361, 0.114583)$、$\alpha_3 = (0.004659, 0.010399)$，加總 = 0.015058。
3. **留給你**：改寫成 log 空間（TODO 3）。

In [ ]:
# 最小例子：TODO 3（3 次擲骰子）
x_mini = np.array([5, 5, 0])  # 點數 6, 6, 1

# ---- 1. 正確答案：暴力列舉 ----
total = 0.0
for path in itertools.product(range(2), repeat=3):
    p = pi_true[path[0]] * B_true[path[0], x_mini[0]]
    for t in range(1, 3):
        p *= A_true[path[t - 1], path[t]] * B_true[path[t], x_mini[t]]
    total += p
    print("".join(STATES[s] for s in path), round(p, 6))
print("P(x) =", round(total, 6), "；ln P(x) =", round(np.log(total), 4))

# ---- 2. 積木：一般空間的 forward ----
alpha = pi_true * B_true[:, x_mini[0]]
print("\nalpha_1 =", alpha.round(6))
for t in range(1, 3):
    alpha = (alpha @ A_true) * B_true[:, x_mini[t]]
    print(f"alpha_{t + 1} =", alpha.round(6))
print("sum =", alpha.sum().round(6))

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應為 -4.1959）：
# print(round(logsumexp(forward_log(x_mini, pi_true, A_true, B_true)[-1]), 4))

In [ ]:
def forward_log(x: np.ndarray, pi: np.ndarray, A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """回傳 ln α，shape (T, K)。ln P(x) = logsumexp(la[-1])。"""
    T, K = len(x), len(pi)
    lA, lB = np.log(A), np.log(B)
    la = np.zeros((T, K))
    la[0] = np.log(pi) + lB[:, x[0]]
    for t in range(1, T):
        # TODO 3（程式）：一行（下標公式）
        raise NotImplementedError("TODO 3：forward")
    return la


def forward_naive(x: np.ndarray, pi: np.ndarray, A: np.ndarray, B: np.ndarray) -> float:
    """一般空間（不取 log）—— 用來示範 underflow。"""
    a = pi * B[:, x[0]]
    for t in range(1, len(x)):
        a = (a @ A) * B[:, x[t]]
    return a.sum()


from hmmlearn.hmm import CategoricalHMM


def hmmlearn_model(pi: np.ndarray, A: np.ndarray, B: np.ndarray) -> CategoricalHMM:
    """用給定的參數建一個 hmmlearn 模型（不訓練）。"""
    m = CategoricalHMM(len(pi), init_params="", params="", n_features=B.shape[1])
    m.startprob_, m.transmat_, m.emissionprob_ = pi, A, B
    return m


ref = hmmlearn_model(pi_true, A_true, B_true)
la = forward_log(x_obs, pi_true, A_true, B_true)
ll = logsumexp(la[-1])
print(f"✅ ln P(x)：手寫 {ll:.6f}、hmmlearn {ref.score(x_obs[:, None]):.6f}")
for L in [100, 300, 500, 1000, 2000]:
    print(f"   T = {L:>4}：一般空間 P(x) = {forward_naive(x_obs[:L], pi_true, A_true, B_true):.3e}；log 空間 ln P(x) = {logsumexp(forward_log(x_obs[:L], pi_true, A_true, B_true)[-1]):.1f}")

### 3B. backward 與後驗

$\beta_t(j) = P(x_{t+1:T} \mid z_t = j)$：「從第 $t$ 步的狀態 $j$ 出發，看到剩下觀測」的機率。從尾巴往回算：
$$\beta_T(j) = 1, \qquad \beta_t(j) = \sum_k A_{jk} B_{k, x_{t+1}}\beta_{t+1}(k)$$

**後驗**（forward-backward）：
$$\gamma_t(k) = P(z_t = k \mid x_{1:T}) = \frac{\alpha_t(k)\,\beta_t(k)}{P(x_{1:T})}$$

直覺：$\alpha$ 帶來「過去」的證據、$\beta$ 帶來「未來」的證據。第二節的單步貝氏只用了 $x_t$ 自己。

**檢查**：對每個 $t$，$\sum_k \alpha_t(k)\beta_t(k) = P(x)$（都一樣）。

#### 符號表（3B）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $\ln\beta$ | backward（log） | $(T, K)$ | `lb` |
| $\gamma$ | 後驗 | $(T, K)$ | `gamma` |

**下標公式**：
- TODO 4：$\ln\beta_t(j) = \text{logsumexp}_k(\ln A_{jk} + \ln B_{k, x_{t+1}} + \ln\beta_{t+1}(k))$ → `logsumexp(lA + lB[:, x[t + 1]] + lb[t + 1], axis=1)`（`lA` 是 $(K, K)$、後兩項是 $(K,)$，廣播到每一列；對 $k$ = `axis=1` 加總）。
- TODO 5：$\ln\gamma = \ln\alpha + \ln\beta - \ln P(x)$ → `np.exp(la + lb - ll)`。

In [ ]:
def backward_log(x: np.ndarray, pi: np.ndarray, A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """回傳 ln β，shape (T, K)。"""
    T, K = len(x), len(pi)
    lA, lB = np.log(A), np.log(B)
    lb = np.zeros((T, K))  # ln β_T = ln 1 = 0
    for t in range(T - 2, -1, -1):
        # TODO 4（程式）：一行（下標公式）
        raise NotImplementedError("TODO 4：backward")
    return lb


def posterior(x: np.ndarray, pi: np.ndarray, A: np.ndarray, B: np.ndarray) -> tuple[np.ndarray, float]:
    """回傳 (γ (T, K), ln P(x))。"""
    la = forward_log(x, pi, A, B)
    lb = backward_log(x, pi, A, B)
    ll = logsumexp(la[-1])
    # TODO 5（程式）：gamma
    raise NotImplementedError("TODO 5：後驗")
    return gamma, ll


gamma, ll = posterior(x_obs, pi_true, A_true, B_true)
lb = backward_log(x_obs, pi_true, A_true, B_true)
print("✅ 每個 t 的 logsumexp(ln α_t + ln β_t) 都等於 ln P(x)？", np.allclose(logsumexp(la + lb, axis=1), ll))
print("✅ γ 每一列加總 1？", np.allclose(gamma.sum(1), 1))
print(f"✅ 後驗跟 hmmlearn predict_proba 的最大差異：{np.abs(gamma - ref.predict_proba(x_obs[:, None])).max():.1e}（應 < 1e-8）")
print("最小例子的 P(L | x)：", posterior(x_mini, pi_true, A_true, B_true)[0][:, 1].round(4), "（暴力列舉：[0.8421 0.8117 0.6906]）")

fig, ax = plt.subplots(figsize=(15, 3))
ax.fill_between(np.arange(400), z_true[:400], step="mid", alpha=0.25, color="tab:red", label="truth: L")
ax.plot(gamma[:400, 1], lw=1, label="posterior P(L | x)")
ax.plot(post_single[:400], lw=0.6, alpha=0.6, label="single-roll Bayes")
ax.legend(loc="upper right")
plt.show()

### 3C. Viterbi：最可能的整條路徑

把 forward 的「加總」換成「取最大」：
$$\delta_1(k) = \pi_k B_{k, x_1}, \qquad \delta_t(k) = \max_j\big[\delta_{t-1}(j) A_{jk}\big] B_{k, x_t}, \qquad \text{bp}_t(k) = \arg\max_j\big[\delta_{t-1}(j) A_{jk}\big]$$

最後從 $\arg\max_k\delta_T(k)$ 開始，沿著 $\text{bp}$ 倒著走回去。（log 空間：乘法變加法，`max` 不變。）

**跟後驗解碼的差別**：後驗解碼每一步各自取 $\arg\max_k \gamma_t(k)$ —— 每一步最可能，但**合起來的路徑可能不合法**（例如 $A_{jk} = 0$ 的轉移）；Viterbi 保證是**一整條**最可能的路徑。

#### 符號表（3C）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $\ln\delta$ | 走到 $(t, k)$ 的最佳路徑 log 機率 | $(T, K)$ | `ld` |
| bp | 回溯指標 | $(T, K)$ | `bp` |
| $\hat z$ | 最佳路徑 | $(T,)$ | `path` |

**下標公式**：
- TODO 6：`s = ld[t - 1][:, None] + lA`（$(K, K)$，第 $(j, k)$ 個）；`bp[t] = s.argmax(axis=0)`；`ld[t] = s.max(axis=0) + lB[:, x[t]]`；回溯 `path[t - 1] = bp[t, path[t]]`。

#### 最小例子：TODO 6

同樣 $x = (6, 6, 1)$：暴力列舉的最大路徑是 **LLL**（0.010125，見 3A 的表）。注意第 3 次擲出 1，單步貝氏會說 F，但 Viterbi 考慮「換骰子要付出 $A_{LF} = 0.1$ 的代價」，所以還是 L。後驗 $P(L \mid x)$ 在第 3 步是 0.69 —— 也說 L。

In [ ]:
def viterbi_log(x: np.ndarray, pi: np.ndarray, A: np.ndarray, B: np.ndarray) -> tuple[np.ndarray, float]:
    """回傳 (最佳路徑 (T,), 它的 ln P(x, z))。"""
    T, K = len(x), len(pi)
    lA, lB = np.log(A), np.log(B)
    ld = np.zeros((T, K))
    bp = np.zeros((T, K), dtype=int)
    ld[0] = np.log(pi) + lB[:, x[0]]
    path = np.zeros(T, dtype=int)
    # TODO 6（程式）：遞推 + 回溯（下標公式）
    raise NotImplementedError("TODO 6：Viterbi")
    return path, float(ld[-1].max())


path_mini, _ = viterbi_log(x_mini, pi_true, A_true, B_true)
print("最小例子：", "".join(STATES[s] for s in path_mini), "（應為 LLL）")

# 暴力列舉對照：隨機 30 段長度 10 的序列
ok = 0
for s in range(30):
    xs = x_obs[s * 10 : s * 10 + 10]
    best = max(itertools.product(range(2), repeat=10),
               key=lambda p: np.log(pi_true[p[0]] * B_true[p[0], xs[0]]) + sum(np.log(A_true[p[t - 1], p[t]] * B_true[p[t], xs[t]]) for t in range(1, 10)))
    ok += np.array_equal(viterbi_log(xs, pi_true, A_true, B_true)[0], best)
print(f"✅ 暴力列舉（2^10 條路徑）對照：{ok}/30 段一樣")

path, lp = viterbi_log(x_obs, pi_true, A_true, B_true)
lp_ref, path_ref = ref.decode(x_obs[:, None], algorithm="viterbi")
print(f"✅ 跟 hmmlearn 的 Viterbi：ln P 差 {abs(lp - lp_ref):.1e}；路徑不同的步數 {int((path != path_ref).sum())}")

In [ ]:
# TODO 7（問答）TODO 3–6
#
# (1) 最小例子：
#     (a) 8 條路徑裡，為什麼 LLL 比 FFF 大？第 3 次擲出 1，為什麼最佳路徑還是 L？
#     (b) 一般空間的 α 到 T 多大就變成 0？用「每一步大約乘上多少」估算：ln P(x) / T ≈ ?
# 答：
#
# (2) 後驗的圖：P(L | x) 在真正的 L 段開始時，是「馬上」升高還是慢慢升高？在 L 段結束時呢？為什麼？
#     （提示：α 只看過去、β 只看未來，γ 兩個都看）
# 答：
#
# (3) 複雜度：forward 是 O(T K²)。如果 K = 9（NER 的標籤數）、T = 40（一句話），一次 forward 大約幾次運算？暴力列舉要列幾條路徑？
# 答：

### 3D. Baum-Welch：只看觀測，學參數（EM）

不知道 $\pi, A, B$，也不知道 $z$。跟第十五章 GMM 的 EM 一樣：

**E-step**（用目前的參數）：
- $\gamma_t(k) = P(z_t = k \mid x)$（3B）；
- $\xi_t(j, k) = P(z_t = j, z_{t+1} = k \mid x) = \dfrac{\alpha_t(j)\,A_{jk}\,B_{k, x_{t+1}}\,\beta_{t+1}(k)}{P(x)}$（「第 $t$ 步從 $j$ 轉到 $k$」的後驗）。

**M-step**（「加權數數」）：
$$\pi_k = \gamma_1(k), \qquad A_{jk} = \frac{\sum_t \xi_t(j, k)}{\sum_t\sum_{k'}\xi_t(j, k')}, \qquad B_{kv} = \frac{\sum_{t: x_t = v}\gamma_t(k)}{\sum_t \gamma_t(k)}$$

如果知道 $z$（監督式），$\gamma$、$\xi$ 就是 0/1，M-step 就是**直接數數**（第 5 節的 NER）。

#### 符號表（3D）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $\ln\xi$ | 轉移的後驗（log） | $(T - 1, K, K)$ | `lxi` |
| $\sum_t\xi_t$ | 期望轉移次數 | $(K, K)$ | `xi_sum` |

**下標公式**：
- TODO 8：$\ln\xi_t(j, k) = \ln\alpha_t(j) + \ln A_{jk} + \ln B_{k, x_{t+1}} + \ln\beta_{t+1}(k) - \ln P(x)$ → `la[:-1, :, None] + lA[None] + (lB[:, x[1:]].T + lb[1:])[:, None, :] - ll`；
  $A$ = `xi_sum / xi_sum.sum(1, keepdims=True)`；$B_{kv}$：對每個 $v$，`gamma[x == v].sum(0)`，再每一列正規化。

In [ ]:
def baum_welch_step(x: np.ndarray, pi: np.ndarray, A: np.ndarray, B: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    """一次 EM。回傳 (新 π, 新 A, 新 B, 舊參數下的 ln P(x))。"""
    la = forward_log(x, pi, A, B)
    lb = backward_log(x, pi, A, B)
    ll = logsumexp(la[-1])
    gamma = np.exp(la + lb - ll)
    lA, lB = np.log(A), np.log(B)
    # TODO 8（程式）：ξ 與 M-step（下標公式）
    raise NotImplementedError("TODO 8：Baum-Welch")
    return gamma[0], A_new, B_new, float(ll)


def baum_welch(x: np.ndarray, pi: np.ndarray, A: np.ndarray, B: np.ndarray, n_iter: int = 300, tol: float = 1e-4) -> dict:
    hist = []
    for _ in range(n_iter):
        with np.errstate(divide="ignore"):  # 學到的機率可能剛好是 0 → log 0 = −inf（合法）
            pi, A, B, ll = baum_welch_step(x, pi, A, B)
        hist.append(ll)
        if len(hist) > 1 and hist[-1] - hist[-2] < tol:
            break
    return {"pi": pi, "A": A, "B": B, "history": hist}


rng = np.random.default_rng(1)
pi0 = np.array([0.6, 0.4])
A0 = np.array([[0.8, 0.2], [0.3, 0.7]])
B0 = rng.dirichlet(np.ones(6), size=2)
mine = baum_welch(x_obs, pi0, A0, B0, n_iter=30, tol=-np.inf)
hl = CategoricalHMM(2, n_iter=30, tol=-np.inf, init_params="", params="ste", n_features=6)
hl.startprob_, hl.transmat_, hl.emissionprob_ = pi0, A0, B0
hl.fit(x_obs[:, None])
print(f"✅ 同樣的初始值、30 次 EM：A 的最大差異 {np.abs(mine['A'] - hl.transmat_).max():.1e}、B {np.abs(mine['B'] - hl.emissionprob_).max():.1e}（應 < 1e-8）")
print("✅ log-likelihood 每一步都沒有下降？", bool(np.all(np.diff(mine["history"]) > -1e-9)))

# 跑到收斂：EM 對初始值很敏感（實驗 3）。上面的隨機 A0 容易卡在局部最佳，
# 這裡用一個「合理的猜測」當起點：骰子不常換（A 的對角線大）、兩顆骰子都接近公正（B 接近均勻、但不完全一樣）。
bw = baum_welch(x_obs, np.array([0.5, 0.5]), np.array([[0.9, 0.1], [0.2, 0.8]]), np.random.default_rng(4).dirichlet(np.ones(6) * 10, size=2))
print(f"\n從「骰子不常換」的初始值跑到收斂（log-likelihood 進步 < 1e-4）：{len(bw['history'])} 次；最終 ln P(x) = {bw['history'][-1]:.1f}")
print("A =\n", bw["A"].round(3), "\nB =\n", bw["B"].round(3))

### 3E. 實驗

| 實驗 | 問題 |
| --- | --- |
| 1 | Viterbi vs 後驗解碼 vs baseline：準確率；後驗解碼的路徑「合法」嗎？ |
| 2 | Baum-Welch 需要多少資料？$T$ = 200、1,000、5,000 時，學到的參數離真相多遠？ |
| 3 | 不同的初始值：局部最佳、標籤互換 |
| 4 | 狀態數：$K$ = 1、2、3、4 的 BIC（第十五章） |

In [ ]:
# 實驗 1：解碼方式
pred_post = gamma.argmax(axis=1)
dec_tbl = pd.concat([base_tbl, pd.DataFrame({"accuracy": {
    "posterior decoding (true params)": (pred_post == z_true).mean(),
    "Viterbi (true params)": (path == z_true).mean(),
    "Viterbi (Baum-Welch params)": (viterbi_log(x_obs, bw["pi"], bw["A"], bw["B"])[0] == z_true).mean(),
}})])
display(dec_tbl.round(4))
print("後驗解碼的「換骰子」次數：", int((np.diff(pred_post) != 0).sum()), "；Viterbi：", int((np.diff(path) != 0).sum()), "；真相：", int((np.diff(z_true) != 0).sum()))

# 一個「非法路徑」的例子：如果 A 不允許 L → F（只能 L → 中間狀態 → F），後驗解碼會怎樣？
A3 = np.array([[0.9, 0.1, 0.0], [0.0, 0.9, 0.1], [0.5, 0.0, 0.5]])
B3 = np.array([[0.8, 0.2], [0.2, 0.8], [0.5, 0.5]])
pi3 = np.array([1.0, 0.0, 0.0])
_, x3 = sample_hmm(400, pi3, A3, B3, np.random.default_rng(3))
with np.errstate(divide="ignore"):
    g3 = posterior(x3, pi3, A3, B3)[0].argmax(1)
    v3 = viterbi_log(x3, pi3, A3, B3)[0]
illegal = lambda p: int(sum(A3[a, b] == 0 for a, b in zip(p[:-1], p[1:])))
print(f"\n3 狀態、有禁止轉移的 HMM：後驗解碼有 {illegal(g3)} 個非法轉移；Viterbi 有 {illegal(v3)} 個")

In [ ]:
# 實驗 2：資料量（從真相附近的隨機初始值開始；每個 T 各 3 次取最好的 log-likelihood）
def align(res: dict) -> dict:
    """標籤互換：讓「6 點機率比較大」的狀態排第 2 個（L）。"""
    if res["B"][0, 5] > res["B"][1, 5]:
        o = [1, 0]
        res = {**res, "pi": res["pi"][o], "A": res["A"][np.ix_(o, o)], "B": res["B"][o]}
    return res


rows = []
for L in [200, 1000, 5000]:
    zL, xL = sample_hmm(L, pi_true, A_true, B_true, np.random.default_rng(10 + L))
    best = None
    for s in range(3):
        r = np.random.default_rng(s)
        res = baum_welch(xL, r.dirichlet([5, 5]), r.dirichlet([5, 5], size=2), r.dirichlet(np.ones(6) * 5, size=2))
        if best is None or res["history"][-1] > best["history"][-1]:
            best = res
    best = align(best)
    rows.append({"T": L, "A_FF": best["A"][0, 0], "A_LL": best["A"][1, 1], "B_L(6)": best["B"][1, 5], "max |A − A_true|": np.abs(best["A"] - A_true).max(), "max |B − B_true|": np.abs(best["B"] - B_true).max(), "EM iters": len(best["history"])})
display(pd.DataFrame(rows).set_index("T").round(3))
print("真相：A_FF = 0.95、A_LL = 0.90、B_L(6) = 0.5")

In [ ]:
# 實驗 3：10 組隨機初始值（T = 2000）
rows = []
for s in range(10):
    r = np.random.default_rng(100 + s)
    res = baum_welch(x_obs, r.dirichlet([1, 1]), r.dirichlet([1, 1], size=2), r.dirichlet(np.ones(6), size=2))
    rows.append({"seed": s, "final log-lik": res["history"][-1], "iters": len(res["history"]), "state 0: P(6)": res["B"][0, 5], "state 1: P(6)": res["B"][1, 5], "A_00": res["A"][0, 0], "A_11": res["A"][1, 1]})
display(pd.DataFrame(rows).set_index("seed").round(3))

In [ ]:
# 實驗 4：狀態數 K 的 BIC（hmmlearn，每個 K 跑 5 個初始值取最好）
rows = []
for K in [1, 2, 3, 4]:
    best = max((CategoricalHMM(K, n_iter=500, random_state=s, n_features=6).fit(x_obs[:, None]) for s in range(5)), key=lambda m: m.score(x_obs[:, None]))
    n_params = (K - 1) + K * (K - 1) + K * 5
    rows.append({"K": K, "log-lik": best.score(x_obs[:, None]), "# params": n_params, "BIC": best.bic(x_obs[:, None])})
display(pd.DataFrame(rows).set_index("K").round(1))

In [ ]:
# TODO 9（問答）實驗 1–4
#
# (1) 實驗 1：
#     (a) Viterbi 和後驗解碼的準確率差多少？比最好的滑動視窗好多少？用 Baum-Welch 學到的參數，準確率掉多少？
#     (b) 「換骰子」的次數：誰比較接近真相？後驗解碼為什麼會換得比較頻繁？
#     (c) 3 狀態的例子：後驗解碼產生了非法轉移嗎？什麼時候你一定要用 Viterbi？什麼時候後驗比較有用？（提示：要「機率」還是要「一條路徑」）
# 答：
#
# (2) 實驗 2：T 要多大，A、B 才估得準（誤差 < 0.05）？200 次擲骰子大約只有幾段 L？為什麼 A 比 B 難估？
# 答：
#
# (3) 實驗 3：
#     (a) 10 組初始值的最終 log-likelihood 都一樣嗎？有幾組卡在比較差的局部最佳？（iters = 300 代表跑到上限還沒收斂 —— 卡在很平的地方）
#     (b) 有沒有「狀態 0 才是灌鉛骰子」的情況？這是錯嗎？（第十五章的 label switching）
# 答：
#
# (4) 實驗 4：BIC 選了 K = 幾？跟真相（K = 2）一樣嗎？
#     (a) K = 1 → 2 的 log-likelihood 進步多少？BIC 的懲罰多了多少（多 8 個參數 × ln(2000) / 2）？
#     (b) 如果結果不是 2：這代表 HMM 錯了，還是「2,000 次擲骰子的證據不夠強」？T 變成 20,000 時，你預期會選幾？（提示：log-likelihood 的進步 ∝ T，懲罰 ∝ ln T）
#     (c) K = 3、4 的 log-likelihood 更高，BIC 為什麼不選它們？
# 答：

---
## 4. 套件實作：`hmmlearn`

| 類別 | 觀測 | 例子 |
| --- | --- | --- |
| `CategoricalHMM` | 離散（0…V−1） | 骰子、字 |
| `GaussianHMM` | 連續，每個狀態一個高斯（`covariance_type` 同第十五章） | 股價報酬的高／低波動期 |
| `GMMHMM` | 每個狀態一個 GMM | 語音特徵 |

| 參數／方法 | 意思 |
| --- | --- |
| `n_iter`、`tol` | EM 的次數上限、收斂門檻 |
| `init_params`、`params` | 哪些參數要隨機初始化、哪些要更新（`"s"` π、`"t"` A、`"e"` B；Gaussian 是 `"m"` 平均、`"c"` 共變異） |
| `fit(X, lengths)` | 多條序列接在一起，`lengths` 給每條的長度 |
| `decode`、`predict`、`predict_proba`、`score` | Viterbi、Viterbi 路徑、後驗、ln P(x) |
| `bic(X)` | BIC |

下面用 `GaussianHMM` 做一個連續的例子：模擬「平靜期／動盪期」的日報酬率（兩個狀態的標準差 1% vs 3%），看 HMM 能不能找出動盪期。（第六部分的 GARCH 也在建模「波動率隨時間變化」，但方式不同。）

In [ ]:
from hmmlearn.hmm import GaussianHMM

A_vol = np.array([[0.98, 0.02], [0.05, 0.95]])
rng = np.random.default_rng(7)
z_vol = np.zeros(1500, dtype=int)
for t in range(1, 1500):
    z_vol[t] = rng.choice(2, p=A_vol[z_vol[t - 1]])
ret = rng.normal(0.0005, np.where(z_vol == 0, 0.01, 0.03))

gh = GaussianHMM(2, covariance_type="diag", n_iter=200, random_state=0).fit(ret[:, None])
hi = int(np.argmax(gh.covars_.ravel()))  # 變異比較大的狀態 = 動盪期
z_hat = (gh.predict(ret[:, None]) == hi).astype(int)
print("學到的標準差：", np.sqrt(gh.covars_.ravel()).round(4), "；學到的 A：\n", gh.transmat_.round(3))
print(f"動盪期的辨識準確率：{(z_hat == z_vol).mean():.3f}；如果只用「|報酬| > 2%」：{((np.abs(ret) > 0.02).astype(int) == z_vol).mean():.3f}")

fig, axes = plt.subplots(2, 1, figsize=(15, 4), sharex=True)
axes[0].plot(ret, lw=0.5)
axes[0].set_ylabel("daily return")
axes[1].fill_between(np.arange(1500), z_vol, step="mid", alpha=0.3, color="tab:red", label="truth: volatile")
axes[1].plot(gh.predict_proba(ret[:, None])[:, hi], lw=0.8, label="P(volatile | returns)")
axes[1].legend(loc="upper right")
plt.show()

In [ ]:
# TODO 10（問答）
#
# (1) GaussianHMM 學到的兩個標準差、轉移機率，跟真相（0.01、0.03；0.98、0.95）差多少？
# (2) 「|報酬| > 2%」為什麼比 HMM 差很多？動盪期裡也有很多「小」的報酬 —— HMM 用什麼資訊把它們判成動盪期？
# 答：

---
## 5. 監督式 HMM：CoNLL-2003 命名實體辨識（NER）

**任務**：每個字標上 9 種 BIO 標籤之一：`B-X` = 實體 X 的第一個字、`I-X` = 實體 X 的後續字、`O` = 不是實體；X ∈ {PER 人名, ORG 組織, LOC 地點, MISC 其他}。

```
EU     rejects German  call to boycott British lamb .
B-ORG  O       B-MISC  O    O  O       B-MISC  O    O
```

**HMM 標註器**：狀態 = 標籤、觀測 = 字。有標好的訓練資料 → 不需要 EM，**直接數數**（3D 最後一段）：
$$\pi_k \propto \#(\text{句首是 } k) + \alpha, \quad A_{jk} \propto \#(j \to k) + \alpha, \quad B_{kw} \propto \#(\text{標籤 } k \text{ 的字是 } w) + \alpha$$
（$+\alpha$ = 拉普拉斯平滑，第六章。）解碼用 3C 的 Viterbi。

**沒看過的字**（OOV）：測試集裡有很多訓練時沒出現的人名、地名。訓練時把「只出現一次」的字換成特殊符號，測試時沒看過的字也換成同一個符號 → 模型學到「罕見字」的標籤分布。兩種版本：

- `<UNK>`：全部罕見字同一個符號；
- `<UNK:Xxxx>`：依**字形**（大寫 X、小寫 x、數字 d、連續相同字元縮成兩個）分開，例如 `Xxx`、`XX`、`dd.dd` —— 大寫開頭的罕見字很可能是人名或地名。

**評估**：實體層級的 precision／recall／F1（`ml_dl_llm.metrics.ner_metrics`，seqeval）：一個實體的**邊界和類別都對**才算對。成績寫進 `results/conll2003_ner.csv`，第 21 章 CRF 接著比。

#### 符號表（5）

| 符號 | 意義 | 程式裡 |
| --- | --- | --- |
| `vocab` | 字（或 UNK 符號）→ 編號 | `dict` |
| `norm(w)` | 把罕見字／沒看過的字換成 UNK 符號 | 函式 |
| `counts_pi`、`counts_A`、`counts_B` | 數數的結果 | $(K,)$、$(K, K)$、$(K, V)$ |

In [ ]:
import re
from collections import Counter

from ml_dl_llm.data import load_conll2003

ds, label_names = load_conll2003()
K_ner = len(label_names)
train_tokens = [list(ex["tokens"]) for ex in ds["train"]]
train_tags = [list(ex["ner_tags"]) for ex in ds["train"]]
test_tokens = [list(ex["tokens"]) for ex in ds["test"]]
test_tags = [list(ex["ner_tags"]) for ex in ds["test"]]
print("標籤：", label_names)
print("句子數：train", len(train_tokens), "、validation", len(ds["validation"]), "、test", len(test_tokens))
tag_count = Counter(t for s in train_tags for t in s)
print("train 標籤分布：", {label_names[k]: tag_count[k] for k in range(K_ner)})
word_count = Counter(w for s in train_tokens for w in s)
test_words = [w for s in test_tokens for w in s]
print(f"test 的字有 {np.mean([w not in word_count for w in test_words]):.1%} 沒在 train 出現過；"
      f"其中實體字（非 O）的比例：{np.mean([word_count[w] == 0 for s, y in zip(test_tokens, test_tags) for w, t in zip(s, y) if t != 0]):.1%}")


def word_shape(w: str) -> str:
    s = re.sub("[A-Z]", "X", w)
    s = re.sub("[a-z]", "x", s)
    s = re.sub("[0-9]", "d", s)
    return re.sub(r"(.)\1+", r"\1\1", s)  # 連續相同字元縮成兩個：Xxxxx → Xxx


print("字形例子：", {w: word_shape(w) for w in ["EU", "German", "1996-08-22", "McDonald", "3.5", "boycott"]})

#### 最小例子：TODO 11（數數）

兩句話、3 種標籤 {0: O, 1: B-PER, 2: I-PER}：

```
句 1：John  Smith  runs       標籤 1 2 0
句 2：Mary  runs              標籤 1 0
```

1. **正確答案**（$\alpha = 0$）：句首：B-PER 2 次 → $\pi = (0, 1, 0)$。轉移：B→I 1、I→O 1、B→O 1 → $A$ 的 B-PER 列 = $(0.5, 0, 0.5)$、I-PER 列 = $(1, 0, 0)$、O 列沒有資料（0/0）。發射：B-PER 發出 John、Mary 各 1 → 各 0.5。
2. **積木**：`np.add.at`（前置複習 10）；`zip(y[:-1], y[1:])` 是相鄰的標籤對。
3. **留給你**：寫成函式，加上平滑 $\alpha$。$\alpha = 1$ 時，O 列變成 $(1/3, 1/3, 1/3)$。

In [ ]:
# 最小例子：TODO 11（數數）
sents_mini = [["John", "Smith", "runs"], ["Mary", "runs"]]
tags_mini = [[1, 2, 0], [1, 0]]
vocab_mini = {"John": 0, "Smith": 1, "runs": 2, "Mary": 3}

# ---- 1. 正確答案：迴圈 ----
A_cnt = np.zeros((3, 3))
for y_ in tags_mini:
    for a, b in zip(y_[:-1], y_[1:]):
        A_cnt[a, b] += 1
print("轉移次數：\n", A_cnt)

# ---- 2. 積木：np.add.at ----
B_cnt = np.zeros((3, 4))
for s_, y_ in zip(sents_mini, tags_mini):
    np.add.at(B_cnt, (y_, [vocab_mini[w] for w in s_]), 1)
print("發射次數：\n", B_cnt)

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（A 的 O 列應為 [1/3 1/3 1/3]、B-PER 列 [0.5 0 0.5]（α=1 時是 [0.4 0.2 0.4]））：
# p_, A_, B_ = count_hmm(sents_mini, tags_mini, vocab_mini, 3, alpha=1.0, norm=lambda w: w); print(p_.round(3), "\n", A_.round(3), "\n", B_.round(3))

In [ ]:
def count_hmm(sents: list[list[str]], tags: list[list[int]], vocab: dict[str, int], K: int, alpha: float,
              norm=lambda w: w) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """監督式 HMM：數數 + 平滑。回傳 (π (K,), A (K, K), B (K, V))，每一列都已正規化。"""
    V = len(vocab)
    pi = np.full(K, alpha)
    A = np.full((K, K), alpha)
    B = np.full((K, V), alpha)
    # TODO 11（程式）：對每句話
    #   - pi[y[0]] += 1
    #   - 相鄰標籤對 (a, b)：A[a, b] += 1
    #   - np.add.at(B, (y, [vocab[norm(w)] for w in s]), 1)
    #   最後每個都除以自己那一列的總和
    raise NotImplementedError("TODO 11：數數")
    return pi, A, B

In [ ]:
from ml_dl_llm.metrics import ner_metrics
from ml_dl_llm.results import compare, log_result


def make_normalizer(mode: str):
    """mode："unk"（罕見字 → <UNK>）、"shape"（罕見字 → <UNK:字形>）。出現 1 次 = 罕見。"""
    def norm(w: str) -> str:
        if word_count[w] > 1:
            return w
        return "<UNK>" if mode == "unk" else f"<UNK:{word_shape(w)}>"
    return norm


def to_names(seqs: list[list[int]]) -> list[list[str]]:
    return [[label_names[t] for t in s] for s in seqs]


hmm_ner = {}
for mode in ["unk", "shape"]:
    norm = make_normalizer(mode)
    vocab = {w: i for i, w in enumerate(sorted({norm(w) for s in train_tokens for w in s} | {"<UNK>"}))}

    def encode(s: list[str]) -> np.ndarray:
        return np.array([vocab.get(norm(w), vocab["<UNK>"]) for w in s])  # 字形沒看過 → <UNK>

    for alpha in [0.01, 0.1, 1.0]:
        pi_n, A_n, B_n = count_hmm(train_tokens, train_tags, vocab, K_ner, alpha, norm)
        t0 = time.perf_counter()
        pred = [viterbi_log(encode(s), pi_n, A_n, B_n)[0].tolist() for s in test_tokens]
        hmm_ner[(mode, alpha)] = {**ner_metrics(to_names(test_tags), to_names(pred)), "V": len(vocab), "decode_sec": round(time.perf_counter() - t0, 1)}
        if (mode, alpha) == ("shape", 0.1):
            best_pred, best_A = pred, A_n
display(pd.DataFrame(hmm_ner).T)

BENCH = "conll2003_ner"
for (mode, alpha), m in hmm_ner.items():
    if alpha == 0.1:
        log_result(BENCH, f"HMM (counts, UNK={mode}, alpha={alpha})", {k: m[k] for k in ["precision", "recall", "f1", "token_accuracy"]}, split="test")
compare(BENCH, "f1")

In [ ]:
# 學到的轉移矩陣：哪些轉移幾乎不可能？
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(np.log10(best_A), annot=True, fmt=".1f", xticklabels=label_names, yticklabels=label_names, cmap="viridis", ax=ax)
ax.set_title("log10 A (UNK=shape, alpha=0.1): row = from, column = to")
plt.show()

In [ ]:
# TODO 12（問答）
#
# (1) OOV：test 有多少比例的字沒在 train 出現過？實體字呢？為什麼實體字特別容易是 OOV？
# 答：
#
# (2) <UNK> vs <UNK:字形>：F1 差多少？recall 和 precision 哪一個進步比較多？為什麼字形對 NER 這麼有用？
# 答：
#
# (3) 平滑 α：α 大或小，哪個好？B 有上萬欄，α = 1 對「罕見字的發射機率」有什麼影響？（第六章 TODO：平滑太大會把資料淹掉）
# 答：
#
# (4) 轉移矩陣：O → I-PER 的機率是多少？為什麼幾乎是 0？Viterbi 會因此避免哪一種錯誤？
# 答：

---
## 6. 診斷：HMM 標註器錯在哪裡？

依「這個字是否在 train 出現過」與「實體類型」拆開看錯誤率。HMM 是**生成式**模型：$P(x_t \mid z_t)$ 只能用「這個字本身」（或我們手動設計的一個 UNK 符號）。它**不能**同時用「字形」「前一個字」「後一個字」「字尾 -ville」這些互相重疊的特徵 —— 因為要讓 $P(x \mid z)$ 是一個合法的機率分布，重疊的特徵會被重複計算（第六章樸素貝氏的獨立假設問題）。這就是第 21 章 CRF 的出發點。

In [ ]:
gold_flat = np.concatenate(test_tags)
pred_flat = np.concatenate(best_pred)
oov_flat = np.array([word_count[w] == 0 for s in test_tokens for w in s])
diag = pd.DataFrame({"gold": [label_names[t] for t in gold_flat], "correct": gold_flat == pred_flat, "OOV": oov_flat})
diag["type"] = diag["gold"].str.replace("B-", "").str.replace("I-", "")
display(diag.groupby(["type", "OOV"])["correct"].agg(["mean", "size"]).round(3).unstack())
conf = pd.crosstab(pd.Series([label_names[t] for t in gold_flat], name="gold"), pd.Series([label_names[t] for t in pred_flat], name="pred"))
display(conf)

# 幾個錯誤的句子
shown = 0
for s, g, p in zip(test_tokens, test_tags, best_pred):
    if g != p and 6 <= len(s) <= 14 and shown < 4:
        print(" ".join(f"{w}/{label_names[a]}" + ("" if a == b else f"→{label_names[b]}") for w, a, b in zip(s, g, p)))
        shown += 1

In [ ]:
# TODO 13（問答，本章總結）
#
# (1) 錯誤分析：
#     (a) OOV 的字跟看過的字，準確率差多少？哪一種實體類型最難？
#     (b) 混淆矩陣：最常見的錯誤是「實體 → O」（漏掉）、「O → 實體」（多標），還是「類型錯」（PER ↔ ORG）？
#     (c) 看錯誤的句子：要判斷對，你自己用了哪些線索？哪些是 HMM 看不到的？
# 答：
#
# (2) 生成式 vs 判別式：HMM 學 P(x, z) = P(z) P(x | z)。如果想加入「前一個字」「字尾」「是否全大寫」這些特徵，
#     HMM 的發射機率要怎麼改？為什麼會有「重複計算」的問題？（第六章樸素貝氏 vs 第三章邏輯回歸的對比）
# 答：
#
# (3) 第一部分的比較：HMM 跟第十五章 GMM 有什麼一樣、什麼不一樣？（提示：GMM = 「沒有轉移」的 HMM）
# 答：

---
## 7. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

**賭場**（T = 2,000，已知真相）

| 方法 | 做法 | 準確率 |
| --- | --- | --- |
| 單步貝氏 | | |
| 滑動視窗（最好的） | | |
| 後驗解碼（真參數） | forward-backward | |
| Viterbi（真參數） | | |
| Viterbi（Baum-Welch 參數） | | |

**CoNLL-2003 NER**（test）

| 方法 | 做法 | precision | recall | F1 |
| --- | --- | --- | --- | --- |
| HMM（UNK） | 數數 + Viterbi | | | |
| HMM（UNK:字形） | | | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ Rabiner (1989) 的教學論文：scaling 版（不取 log、每步正規化）的 forward-backward
- ☐ Baum-Welch 收斂性；HMM 的可識別性
- ☐ 前向–後向與 belief propagation、因子圖的關係
- ☐ 最大熵馬可夫模型（MEMM）與 label bias 問題 → 第 21 章 CRF

**延伸**
- ☐ 二階 HMM（看前兩個標籤）、Kneser-Ney 等更好的平滑
- ☐ 語音辨識的 GMM-HMM；基因序列（CpG island）
- ☐ Kalman filter（第六部分）：連續狀態的 HMM
- ☐ 用 HMM 做詞性標註（Penn Treebank）